In [22]:
import pandas as pd
import re

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [24]:
download_service = Service()
driver = webdriver.Chrome(service=download_service)

#driver.get("https://www.sklepopon.com/szukaj-opony?sezon=zimowe&rozmiar=205/55R16&ofs=0")
# driver.get("https://www.oponeo.pl/wybierz-opony/s=1/zimowe/t=1/osobowe/r=1/205-55-r16")
# https://www.oponeo.pl/wybierz-opony/s=1/zimowe/t=1/osobowe/r=1/205-55-r16#&&/wEXDAULcGNrX1ByTFNST0YFA+KUpAUVcGNrX0xzdFNTb3J0UGFyYW1ldGVyBQIzNwUOcGNrX1RyTFNTc25JZHMFATIFCXBja19UckxTUgUD4pSkBQdwY2tfQ1BnBQEyBQpwY2tfVHJMU1BUBQQxMTAwBQhwY2tfSU9GUAUCNjgFDHBja19UckxTU0lkcwUDMjUyBQtwY2tfVHJJT25seQUD4pSkBQpwY2tfVHJMU1BGBQMxOTEFDXBja19UckxTVFRJZHMFATIFFXBja19UckxTU29ydERpcmVjdGlvbgUBMtD9aDKZ+9GYcVUKukEj6XnJhNql
#https://www.oponeo.pl/wybierz-opony/s=1/zimowe/t=1/osobowe/r=1/205-55-r16#&&/wEXDAULcGNrX1ByTFNST0YFA+KUpAUVcGNrX0xzdFNTb3J0UGFyYW1ldGVyBQIzNwUOcGNrX1RyTFNTc25JZHMFATIFCXBja19UckxTUgUD4pSkBQdwY2tfQ1BnBQEzBQpwY2tfVHJMU1BUBQQxMTAwBQhwY2tfSU9GUAUCNjgFDHBja19UckxTU0lkcwUDMjUyBQtwY2tfVHJJT25seQUD4pSkBQpwY2tfVHJMU1BGBQMxOTEFDXBja19UckxTVFRJZHMFATIFFXBja19UckxTU29ydERpcmVjdGlvbgUBMlWnHdgz3AaHVlQ3gBlfXfTWethI

sklep_opon_base_url = "https://www.sklepopon.com/szukaj-opony?sezon=zimowe&rozmiar=205/55R16&ofs="
oponeo_base_url = "https://www.oponeo.pl/wybierz-opony/s=1/zimowe/t=1/osobowe/r=1/205-55-r16"

In [26]:

import time

def hide_sklep_opon_popups():
    # Obsługa przycisku akceptacji ciasteczek
    try:
        btn_cookie = driver.find_element(By.CSS_SELECTOR, "#klaro > div > div > div > div > div > button")
        btn_cookie.click()
        print("Przycisk akceptacji ciasteczek został kliknięty.")
    except NoSuchElementException:
        print("Przycisk akceptacji ciasteczek nie został znaleziony.")
    except Exception as e:
        print("Nie udało się kliknąć przycisku akceptacji ciasteczek:", e)
    
    # Obsługa okna powiadomień w shadow DOM
    try:
        driver.execute_script("""
            const shadowHost = document.querySelector("body > div.gr-visual-prompt");
            if (shadowHost) {
                const shadowRoot = shadowHost.shadowRoot;
                const closeButton = shadowRoot.querySelector("div > div:nth-child(2) > button:nth-child(1)");
                if (closeButton) {
                    closeButton.click();
                    console.log("Okienko powiadomień zostało zamknięte.");
                } else {
                    console.log("Nie znaleziono przycisku zamknięcia powiadomień.");
                }
            } else {
                console.log("Okno powiadomień nie zostało znalezione.");
            }
        """)
    except Exception as e:
        print("Nie udało się zamknąć okienka powiadomień:", e)

def close_oponeo_privacy_popup():
    try:
        # Wyszukanie elementu przycisku "Odrzuć wszystkie"
        reject_button = driver.find_element(By.CSS_SELECTOR, "#consentsBar > div.buttonsContainer.container > div > span.reject")
        reject_button.click()
        print("Okienko prywatności zostało zamknięte.")
    except NoSuchElementException:
        print("Okienko prywatności nie jest widoczne lub zostało już zamknięte.")
    except Exception as e:
        print("Wystąpił błąd podczas zamykania okienka prywatności:", e)

In [21]:


def load_sklep_opon_tyre_data():
    try:
        # Znalezienie wszystkich elementów opon w sekcji listing-products-element
        opony_elements = driver.find_elements(By.CSS_SELECTOR, 'div[data-c-name="listing-products-element"]')
        
        # Iteracja przez każdy element opony
        for opona_element in opony_elements:
            # Słownik do przechowywania danych jednej opony
            opona_data = {}
    
            # Pobranie danych do słownika
            opona_data['name'] = opona_element.get_attribute('data-ee-product-properties').split(";")[0].split(":")[1]
            opona_data['price'] = float(opona_element.get_attribute('data-ee-product-properties').split(";")[3].split(":")[1])
            opona_data['brand'] = opona_element.get_attribute('data-ee-product-properties').split(";")[4].split(":")[1]
            opona_data['size'] = opona_element.get_attribute('data-ee-product-properties').split(";")[5].split(":")[1]
            opona_data['model'] = opona_element.get_attribute('data-ee-product-properties').split(";")[6].split(":")[1]
            
            # Pobieranie szczegółowych informacji o etykiecie
            etykieta_elements = opona_element.find_elements(By.CSS_SELECTOR, 'span.icon-fuel-new ~ span, span.icon-rain-new ~ span, span.icon-speaker-new ~ span')
            opona_data['fuel_index'] = etykieta_elements[0].text if len(etykieta_elements) > 0 else None
            opona_data['wet_grip_index'] = etykieta_elements[1].text if len(etykieta_elements) > 1 else None
            opona_data['noise_index'] = etykieta_elements[2].text.split(" ")[0] if len(etykieta_elements) > 2 else None
            
            # Pobieranie poziomu hałasu, uwzględniając wewnętrzny <span> z dB
            try:
                noise_level_elements = opona_element.find_elements(By.CSS_SELECTOR, "span.self-center.tracking-tighter.sm\\:tracking-normal")
                for noise_level_element in noise_level_elements:
                    text = noise_level_element.text
                    match = re.search(r'\d+', text)
                    if match:
                        noise_level = int(match.group())
                    else:
                        noise_level = None
                    opona_data['noise_level'] = noise_level
            except (NoSuchElementException, IndexError):
                opona_data['noise_level'] = None
            
            class_mapping = {
                "Premium": "Premium",
                "Średnia": "Średnia",
                "Średniej": "Średnia",
                "Ekonomiczna": "Ekonomiczna",
                "Ekonomicznej": "Ekonomiczna"
            }
            try:
                klasa_element = opona_element.find_element(By.XPATH, ".//*[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'klas')]")
                # Dopasuj i wyczyść tekst
                opona_class = klasa_element.text.lower().replace("w klasie ", "").replace("klasa ", "").strip().capitalize()
                opona_data['class'] = class_mapping.get(opona_class, opona_class)
            except NoSuchElementException:
                opona_data['class'] = None
            
            # Pobieranie oceny użytkownika (tekstowa wartość obok gwiazdek)
            try:
                user_rating_element = opona_element.find_element(By.XPATH, ".//li[contains(@class, 'xl:hidden')]//span[contains(@class, 'ml-1')]")
                opona_data['user_rating'] = float(user_rating_element.text.replace(",", "."))
            except NoSuchElementException:
                opona_data['user_rating'] = None
            
            # Dodajemy dane opony do listy
            sklep_opon_tyres_data.append(opona_data)
    finally:
        pass

sklep_opon_tyres_data = []
offset = 0

while True:
    url = f"{sklep_opon_base_url}{offset}"
    driver.get(url)
    time.sleep(4)
    
    if offset == 0:
        hide_sklep_opon_popups()
    
    load_sklep_opon_tyre_data()
    
    offset += 20
    # Warunek zatrzymania przy pustej stronie
    if len(driver.find_elements(By.CSS_SELECTOR, 'div[data-c-name="listing-products-element"]')) == 0:
        print("Brak nowych danych. Koniec paginacji.")
        break
# Zamknięcie WebDrivera
driver.quit()

# Wydrukowanie listy wszystkich danych o oponach
df = pd.DataFrame(sklep_opon_tyres_data)
display(df)

Przycisk akceptacji ciasteczek nie został znaleziony.
Brak nowych danych. Koniec paginacji.


,name,price,brand,size,model,fuel_index,wet_grip_index,noise_index,noise_level,class,user_rating
0,Ultra Grip Performance 3 205/55 R16 91 T,446.98,Goodyear,205/55 R16,Ultra Grip Performance 3,C,B,B,70.0,Premium,5.5
1,Winguard Snow'G WH2 205/55 R16 91 H,310.00,Nexen,205/55 R16,Winguard Snow'G WH2,D,C,B,70.0,Średnia,5.3
2,Frigo HP2 205/55 R16 91 H,283.00,Dębica,205/55 R16,Frigo HP2,C,C,B,72.0,Ekonomiczna,5.2
3,Frigo 2 205/55 R16 91 T,239.00,Dębica,205/55 R16,Frigo 2,C,C,B,71.0,Ekonomiczna,5.1
4,Winter i*cept RS3 W462 205/55 R16 91 T,322.00,Hankook,205/55 R16,Winter i*cept RS3 W462,C,B,B,72.0,Premium,5.3
...,...,...,...,...,...,...,...,...,...,...,...
415,Ultra Grip 8 205/55 R16 91 T,374.43,Goodyear,205/55 R16,Ultra Grip 8,D,D,B,71.0,Premium,5.2
416,Blizzak LM005 205/55 R16 94 V,637.89,Bridgestone,205/55 R16,Blizzak LM005,C,A,B,71.0,Premium,5.5
417,WINTERPRO2 (EVO)* 205/55 R16 91 T,322.82,Gt radial,205/55 R16,WINTERPRO2 (EVO)*,D,B,B,70.0,None,NaN
418,Winter Sport 5 205/55 R16 94 H,464.38,Dunlop,205/55 R16,Winter Sport 5,C,B,B,71.0,Premium,5.4


In [27]:
driver.get(oponeo_base_url)

close_oponeo_privacy_popup()



Okienko prywatności zostało zamknięte.
